In [2]:
import re

def clean_text(text):
    """Basic cleaning that preserves useful scientific punctuation."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_pages(pages, chunk_size=500, overlap=80):
    """
    Split pages into word-based chunks.
    Chunk size and overlap are measured in words, not tokens.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size.")

    chunks = []
    chunk_number = 0
    step = chunk_size - overlap

    for page_data in pages:
        text = clean_text(page_data["text"])
        words = text.split()

        for start in range(0, len(words), step):
            selected = words[start:start + chunk_size]

            if not selected:
                continue

            chunk_number += 1

            chunks.append({
                "document": page_data["document"],
                "page": page_data["page"],
                "chunk_id": f"chunk_{chunk_number}",
                "text": " ".join(selected)
            })

            if start + chunk_size >= len(words):
                break

    return chunks
